# 05 - Compute Metrics Batch

This notebook computes metrics for inferred runs with two resilience features:
- **Incremental persistence**: each successful run is written immediately to `metrics_master.csv`
- **Resume support**: reruns skip already-computed `run_id`s unless overwrite is enabled

If the kernel crashes, rerun the notebook and it continues from the last saved run.


In [6]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import insitucnv as icv

In [7]:
ROOT = Path('/home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains')
RUNS_MANIFEST = ROOT / 'manifests' / 'runs.csv'
RUNS_DIR = ROOT / 'data' / 'runs'
METRICS_OUT = ROOT / 'results' / 'metrics' / 'metrics_master.csv'
STATUS_LOG = ROOT / 'results' / 'logs' / 'run_status.csv'

METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)

assert RUNS_MANIFEST.exists(), f'Missing runs manifest: {RUNS_MANIFEST}'

In [8]:
RUN_ONLY_INFERRED = True
OVERWRITE_EXISTING_ROWS = False  # False = resume mode, True = recompute all candidate run_ids
SAVE_EVERY_N = 1  # keep at 1 for maximum crash safety

METRIC_COLUMNS = [
    'run_id','template_id','count_fraction','gene_panel','n_genes','mean_countsxcell',
    'nmi','ari','F1_all','F1_gain','F1_loss','AUC_gains','PR_gains','AUC_loss','PR_loss'
]

def log_status(run_id, template_id, count_fraction, gene_panel, stage, status, duration_sec=0.0, message=''):
    now = pd.Timestamp.now()
    row = pd.DataFrame([{
        'run_id': run_id,
        'template_id': template_id,
        'count_fraction': count_fraction,
        'gene_panel': gene_panel,
        'stage': stage,
        'status': status,
        'start_time': now,
        'end_time': now,
        'duration_sec': duration_sec,
        'message': message,
    }])
    if STATUS_LOG.exists():
        row.to_csv(STATUS_LOG, mode='a', header=False, index=False)
    else:
        row.to_csv(STATUS_LOG, index=False)


def to_dense_1d(x):
    if sparse.issparse(x):
        return np.asarray(x.todense()).ravel()
    return np.asarray(x).ravel()


def get_inferred_scores(adata):
    # Preferred location from infercnvpy with calculate_gene_values=True
    if 'gene_values_cnv' in adata.layers:
        return to_dense_1d(adata.layers['gene_values_cnv'])

    # fallback to X if needed
    return to_dense_1d(adata.X)


def get_gt_labels(adata):
    if 'CNV_GT' not in adata.layers:
        raise KeyError('CNV_GT layer missing in adata.')
    return to_dense_1d(adata.layers['CNV_GT'])


def compute_cnv_metrics(adata):
    """Compute F1 metrics for all CNV, gain, and loss classes."""
    simulated = get_gt_labels(adata)
    inferred = get_inferred_scores(adata)

    # remove NaNs from inferred if present
    valid = ~np.isnan(inferred)
    simulated = simulated[valid]
    inferred = inferred[valid]

    # Binary masks
    true_cnv = np.mean(inferred[simulated != 0] != 0) if np.sum(simulated != 0) > 0 else 0.0
    false_neutrals = np.mean(inferred[simulated == 0] != 0) if np.sum(simulated == 0) > 0 else 0.0
    true_neutrals = 1 - false_neutrals
    false_cnv = 1 - true_cnv

    true_gains = np.mean(inferred[simulated == 1] > 0) if np.sum(simulated == 1) > 0 else 0.0
    true_losses = np.mean(inferred[simulated == -1] < 0) if np.sum(simulated == -1) > 0 else 0.0

    m_all = icv.tl.compute_performance_metrics(true_cnv, false_neutrals, true_neutrals, false_cnv)
    m_gain = icv.tl.compute_performance_metrics(true_gains, false_neutrals, true_neutrals, 1 - true_gains)
    m_loss = icv.tl.compute_performance_metrics(true_losses, false_neutrals, true_neutrals, 1 - true_losses)

    return {
        'F1_all': float(m_all['F1 Score']),
        'F1_gain': float(m_gain['F1 Score']),
        'F1_loss': float(m_loss['F1 Score']),
    }


def compute_cnv_auc_metrics(adata):
    simulated = get_gt_labels(adata)
    inferred = get_inferred_scores(adata)

    valid = ~np.isnan(inferred)
    simulated = simulated[valid]
    inferred = inferred[valid]

    auc_g, pr_g = icv.tl.compute_auc(simulated, inferred, class_value=1)
    auc_l, pr_l = icv.tl.compute_auc(simulated, inferred, class_value=-1)

    return {
        'AUC_gains': float(auc_g),
        'PR_gains': float(pr_g),
        'AUC_loss': float(auc_l),
        'PR_loss': float(pr_l),
    }


def get_cluster_key(adata):
    # Prefer explicit resolution key if present
    keys = [k for k in adata.obs.columns if str(k).startswith('cnv_leiden_res')]
    if len(keys) > 0:
        return keys[0]
    if 'cnv_leiden' in adata.obs.columns:
        return 'cnv_leiden'
    raise KeyError('No cnv_leiden clustering key found in adata.obs.')


## Select Candidate Runs
Filter `runs.csv` to runs eligible for metric computation.


In [9]:
runs_df = pd.read_csv(RUNS_MANIFEST)

if RUN_ONLY_INFERRED and 'status' in runs_df.columns:
    cand = runs_df[runs_df['status'].astype(str).str.lower() == 'inferred'].copy()
else:
    cand = runs_df.copy()

print(f'Candidate runs for metrics: {len(cand)}')
cand[['run_id', 'template_id', 'count_fraction', 'gene_panel', 'status']].head()

Candidate runs for metrics: 609


,run_id,template_id,count_fraction,gene_panel,status
0,T01_C1_G1000,T01,1,1000,inferred
1,T01_C1_G10000,T01,1,10000,inferred
2,T01_C1_G15000,T01,1,15000,inferred
3,T01_C1_G20000,T01,1,20000,inferred
4,T01_C1_G500,T01,1,500,inferred


## Compute Metrics With Checkpointing
This section writes to `metrics_master.csv` during the loop and skips completed `run_id`s on restart.


In [10]:
# Load existing metrics table (resume support)
if METRICS_OUT.exists():
    metrics_df = pd.read_csv(METRICS_OUT)
else:
    metrics_df = pd.DataFrame(columns=METRIC_COLUMNS)

for col in METRIC_COLUMNS:
    if col not in metrics_df.columns:
        metrics_df[col] = np.nan
metrics_df = metrics_df[METRIC_COLUMNS].copy()

existing_run_ids = set(metrics_df['run_id'].astype(str))
candidate_run_ids = set(cand['run_id'].astype(str))

if OVERWRITE_EXISTING_ROWS:
    # Drop candidate run_ids so they are recomputed from scratch
    metrics_df = metrics_df[~metrics_df['run_id'].astype(str).isin(candidate_run_ids)].copy()
    existing_run_ids = set(metrics_df['run_id'].astype(str))

to_process = cand[~cand['run_id'].astype(str).isin(existing_run_ids)].copy()
print(f'Total candidates: {len(cand)} | Already saved: {len(existing_run_ids & candidate_run_ids)} | To process now: {len(to_process)}')

rows_since_save = 0
processed_rows = 0

for r in to_process.itertuples(index=False):
    run_id = r.run_id
    in_path = RUNS_DIR / f'{run_id}_CNVinf.h5ad'

    if not in_path.exists():
        log_status(run_id, r.template_id, int(r.count_fraction), str(r.gene_panel), 'metrics', 'missing_input', 0.0, f'missing {in_path.name}')
        continue

    t0 = time.time()
    try:
        adata = sc.read_h5ad(in_path)

        cluster_key = get_cluster_key(adata)
        ari = icv.tl.compute_ari(adata, reference_key='simulated_subclone', computed_key=cluster_key)
        nmi = icv.tl.compute_nmi(adata, reference_key='simulated_subclone', computed_key=cluster_key)

        f1 = compute_cnv_metrics(adata)
        auc = compute_cnv_auc_metrics(adata)

        if 'CNV_simulated_raw' in adata.layers:
            mean_counts = float(np.mean(np.array(adata.layers['CNV_simulated_raw'].sum(axis=1)).ravel()))
        elif 'counts' in adata.layers:
            mean_counts = float(np.mean(np.array(adata.layers['counts'].sum(axis=1)).ravel()))
        else:
            mean_counts = np.nan

        row = {
            'run_id': run_id,
            'template_id': r.template_id,
            'count_fraction': int(r.count_fraction),
            'gene_panel': str(r.gene_panel),
            'n_genes': int(adata.n_vars),
            'mean_countsxcell': mean_counts,
            'nmi': float(nmi),
            'ari': float(ari),
            **f1,
            **auc,
        }

        metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
        metrics_df = metrics_df.drop_duplicates(subset=['run_id'], keep='last')

        rows_since_save += 1
        processed_rows += 1

        if rows_since_save >= SAVE_EVERY_N:
            metrics_df = metrics_df.sort_values(['template_id', 'count_fraction', 'gene_panel']).reset_index(drop=True)
            metrics_df.to_csv(METRICS_OUT, index=False)
            rows_since_save = 0

        dt = round(time.time() - t0, 2)
        log_status(run_id, r.template_id, int(r.count_fraction), str(r.gene_panel), 'metrics', 'success', dt, f'cluster_key={cluster_key}')
    except Exception as e:
        dt = round(time.time() - t0, 2)
        log_status(run_id, r.template_id, int(r.count_fraction), str(r.gene_panel), 'metrics', 'failed', dt, str(e))
        print(f'FAILED {run_id}: {e}')

# Final flush
metrics_df = metrics_df.sort_values(['template_id', 'count_fraction', 'gene_panel']).reset_index(drop=True)
metrics_df.to_csv(METRICS_OUT, index=False)

print(f'Processed this run: {processed_rows}')
print(f'Saved metrics: {METRICS_OUT}')
print(f'Total rows in metrics table: {len(metrics_df)}')
metrics_df.head()


Total candidates: 609 | Already saved: 0 | To process now: 609
Processed this run: 609
Saved metrics: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/results/metrics/metrics_master.csv
Total rows in metrics table: 609


,run_id,template_id,count_fraction,gene_panel,n_genes,mean_countsxcell,nmi,ari,F1_all,F1_gain,F1_loss,AUC_gains,PR_gains,AUC_loss,PR_loss
0,T01_C1_G1000,T01,1,1000,1000,28.032335,0.010155,0.010352,0.076275,0.159449,0.000000,0.527417,0.006067,0.510551,0.010201
1,T01_C1_G10000,T01,1,10000,10000,138.865936,0.180122,0.174528,0.247986,0.543705,0.000000,0.667095,0.046968,0.535073,0.012831
2,T01_C1_G15000,T01,1,15000,15000,210.013412,0.277569,0.325056,0.324697,0.663805,0.008886,0.739497,0.111861,0.535388,0.012728
3,T01_C1_G20000,T01,1,20000,20000,287.126190,0.300779,0.317375,0.326175,0.669319,0.012545,0.745061,0.133093,0.534232,0.012769
4,T01_C1_G500,T01,1,500,500,8.569401,0.001510,-0.002626,0.033841,0.045516,0.000000,0.500257,0.007601,0.505290,0.010525


In [11]:
# Reload persisted metrics and show quick integrity checks
metrics_master = pd.read_csv(METRICS_OUT) if METRICS_OUT.exists() else pd.DataFrame(columns=METRIC_COLUMNS)
print(f'Metrics rows on disk: {len(metrics_master)}')
print(f'Unique run_id rows: {metrics_master["run_id"].astype(str).nunique() if len(metrics_master) else 0}')

dups = metrics_master[metrics_master['run_id'].astype(str).duplicated(keep=False)] if len(metrics_master) else pd.DataFrame()
if len(dups) > 0:
    print('WARNING: duplicate run_id rows detected; keeping last per run_id is recommended.')
    display(dups.sort_values('run_id').head(20))

metrics_master.head()


Metrics rows on disk: 609
Unique run_id rows: 609


,run_id,template_id,count_fraction,gene_panel,n_genes,mean_countsxcell,nmi,ari,F1_all,F1_gain,F1_loss,AUC_gains,PR_gains,AUC_loss,PR_loss
0,T01_C1_G1000,T01,1,1000,1000,28.032335,0.010155,0.010352,0.076275,0.159449,0.000000,0.527417,0.006067,0.510551,0.010201
1,T01_C1_G10000,T01,1,10000,10000,138.865936,0.180122,0.174528,0.247986,0.543705,0.000000,0.667095,0.046968,0.535073,0.012831
2,T01_C1_G15000,T01,1,15000,15000,210.013412,0.277569,0.325056,0.324697,0.663805,0.008886,0.739497,0.111861,0.535388,0.012728
3,T01_C1_G20000,T01,1,20000,20000,287.126190,0.300779,0.317375,0.326175,0.669319,0.012545,0.745061,0.133093,0.534232,0.012769
4,T01_C1_G500,T01,1,500,500,8.569401,0.001510,-0.002626,0.033841,0.045516,0.000000,0.500257,0.007601,0.505290,0.010525


In [12]:
# quick grouped summary
if METRICS_OUT.exists():
    m = pd.read_csv(METRICS_OUT)
    display(m.groupby('template_id')[['run_id']].count().rename(columns={'run_id': 'n_runs'}))
    display(m.head())

,n_runs
template_id,
T01,63
T02,62
T03,63
T04,62
T05,63
T06,59
T07,59
T08,62
T09,55


,run_id,template_id,count_fraction,gene_panel,n_genes,mean_countsxcell,nmi,ari,F1_all,F1_gain,F1_loss,AUC_gains,PR_gains,AUC_loss,PR_loss
0,T01_C1_G1000,T01,1,1000,1000,28.032335,0.010155,0.010352,0.076275,0.159449,0.000000,0.527417,0.006067,0.510551,0.010201
1,T01_C1_G10000,T01,1,10000,10000,138.865936,0.180122,0.174528,0.247986,0.543705,0.000000,0.667095,0.046968,0.535073,0.012831
2,T01_C1_G15000,T01,1,15000,15000,210.013412,0.277569,0.325056,0.324697,0.663805,0.008886,0.739497,0.111861,0.535388,0.012728
3,T01_C1_G20000,T01,1,20000,20000,287.126190,0.300779,0.317375,0.326175,0.669319,0.012545,0.745061,0.133093,0.534232,0.012769
4,T01_C1_G500,T01,1,500,500,8.569401,0.001510,-0.002626,0.033841,0.045516,0.000000,0.500257,0.007601,0.505290,0.010525
